# Quick Start: Testing AzureML Models from Databricks

Get up and running in 5 minutes. Test your AzureML deployed models directly from Databricks.

## Step 1: Set Environment Variables

First, configure your AzureML endpoint. In Databricks, use **Admin Settings → Secrets** or set them here:

In [ ]:
import os
import json
import requests
import pandas as pd
import logging

# Latest Azure ML SDK v2
from azure.identity import DefaultAzureCredential, EnvironmentCredential, ChainedTokenCredential
from azure.core.exceptions import HttpResponseError

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration (update with your values)
ENDPOINT_URL = "https://your-endpoint.eastus.inference.ml.azure.com/score"  # Replace with your endpoint
WORKSPACE_NAME = "your-workspace"
SUBSCRIPTION_ID = "your-subscription-id"
RESOURCE_GROUP = "your-resource-group"

print("Configuration set. Endpoint:", ENDPOINT_URL[:50] + "...")
print("✓ Using latest Azure ML SDK v2 (azure-ai-ml)")

## Step 2: Authenticate

In [ ]:
# Get authentication token
credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
token = credential.get_token("https://ml.azure.com/.default").token

print("✓ Authenticated successfully")
print(f"Token valid for ~1 hour")

## Step 3: Test Single Record

In [ ]:
# Prepare a single test record
test_data = {
    "input_data": {
        "columns": ["feature1", "feature2", "feature3"],
        "data": [[1.5, 2.3, 3.1]]  # Your test values
    }
}

# Make request
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(ENDPOINT_URL, headers=headers, json=test_data, timeout=30)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    print(f"✓ Prediction: {response.json()}")
else:
    print(f"✗ Error: {response.text}")

## Step 4: Test with Batch Data

In [ ]:
import numpy as np

# Create sample test data
df_test = pd.DataFrame({
    'feature1': [1.5, 2.0, 1.0, 3.0],
    'feature2': [2.3, 2.5, 2.0, 3.5],
    'feature3': [3.1, 3.5, 3.0, 4.0]
})

print(f"Testing {len(df_test)} records:\n")
display(df_test)

## Step 5: Load Real Data from Spark

# Score the data
# Define which columns are features (exclude target and non-numeric columns)
feature_cols = [col for col in df_pandas.columns if col not in ['target_label', 'customer_id']]

payload = {
    "input_data": {
        "columns": feature_cols,
        "data": df_pandas[feature_cols].fillna(0).values.tolist()
    }
}

response = requests.post(ENDPOINT_URL, headers=headers, json=payload, timeout=60)

if response.status_code == 200:
    df_pandas['prediction'] = response.json()
    
    print(f"✓ Scored {len(df_pandas)} records")
    display(df_pandas[['customer_id'] + feature_cols[:2] + ['prediction']].head())
else:
    print(f"✗ Error: {response.status_code}")
    print(response.text[:200])

def safe_score(data, max_retries=3):
    """
    Score with error handling and retries.
    """
    for attempt in range(max_retries):
        try:
            response = requests.post(
                ENDPOINT_URL,
                headers=headers,
                json=data,
                timeout=30
            )
            
            if response.status_code == 200:
                return response.json(), None
            else:
                error = f"HTTP {response.status_code}: {response.text[:100]}"
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                    continue
                return None, error
                
        except Exception as e:
            error = str(e)
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return None, error
    
    return None, "Max retries exceeded"

# Test the function
test_payload = {
    "input_data": {
        "columns": ['feature1', 'feature2', 'feature3'],
        "data": [[1.5, 2.3, 3.1]]
    }
}

prediction, error = safe_score(test_payload)
if prediction:
    print(f"✓ Prediction: {prediction}")
else:
    print(f"✗ Error: {error}")